# XGBoost Model Training

Tuning XGBoost's hyperparameters with Optuna (optimizing macro F1 on the
validation set), training the final model with the best parameters found,
and evaluating once on the untouched test set. Builds on
notebooks/02_model_selection.ipynb, where XGBoost was selected as the best
candidate with evidence.

In [1]:
import pandas as pd
import numpy as np
from sqlalchemy import create_engine
from dotenv import load_dotenv
from sklearn.preprocessing import LabelEncoder
import os

load_dotenv("../.env")

user = os.environ["POSTGRES_USER"]
password = os.environ["POSTGRES_PASSWORD"]
host = os.environ["POSTGRES_HOST"]
port = os.environ["POSTGRES_PORT"]
db = os.environ["POSTGRES_DB"]
engine = create_engine(f"postgresql+psycopg://{user}:{password}@{host}:{port}/{db}")

train_df = pd.read_sql("SELECT * FROM sensor_readings WHERE split = 'train'", engine)
val_df = pd.read_sql("SELECT * FROM sensor_readings WHERE split = 'val'", engine)

feature_cols = [c for c in train_df.columns if c not in
                ("id", "ahu_id", "reading_time", "labeling", "split")]

X_train = train_df[feature_cols]
X_val = val_df[feature_cols]

le = LabelEncoder()
y_train = le.fit_transform(train_df["labeling"])
y_val = le.transform(val_df["labeling"])

print(f"Train: {X_train.shape}, Val: {X_val.shape}")

Train: (105249, 18), Val: (35083, 18)


## Hyperparameter search

Optuna runs a series of trials, each testing a different combination of
XGBoost hyperparameters, scored by macro F1 on the validation set. The
search space covers the settings that most affect tree-based model
performance: tree depth and count, learning rate, and regularization
strength.

In [2]:
import optuna
import xgboost as xgb
from sklearn.metrics import f1_score

def objective(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 50, 300),
        "max_depth": trial.suggest_int("max_depth", 3, 10),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 1.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-8, 1.0, log=True),
        "random_state": 42,
        "eval_metric": "mlogloss",
    }

    model = xgb.XGBClassifier(**params)
    model.fit(X_train, y_train)
    preds = model.predict(X_val)

    return f1_score(y_val, preds, average="macro")


study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=30, show_progress_bar=True)

print(f"Best macro F1: {study.best_value:.4f}")
print(f"Best params: {study.best_params}")

[I 2026-09-26 17:31:05,202] A new study created in memory with name: no-name-acb734da-ec52-4220-9d6f-23bd1c3cf21f


  0%|          | 0/30 [00:00<?, ?it/s]

[I 2026-09-26 17:31:08,095] Trial 0 finished with value: 0.9870868210090107 and parameters: {'n_estimators': 103, 'max_depth': 9, 'learning_rate': 0.10246954659737768, 'subsample': 0.904006285878292, 'colsample_bytree': 0.8404786961420682, 'reg_alpha': 0.7591175452686056, 'reg_lambda': 0.00020355119824824915}. Best is trial 0 with value: 0.9870868210090107.
[I 2026-09-26 17:31:16,061] Trial 1 finished with value: 0.9884904961227939 and parameters: {'n_estimators': 276, 'max_depth': 9, 'learning_rate': 0.1491451291628595, 'subsample': 0.687658026209894, 'colsample_bytree': 0.8970311357315865, 'reg_alpha': 0.23311219648819345, 'reg_lambda': 2.6294959000069625e-06}. Best is trial 1 with value: 0.9884904961227939.
[I 2026-09-26 17:31:21,473] Trial 2 finished with value: 0.9849326354969562 and parameters: {'n_estimators': 269, 'max_depth': 7, 'learning_rate': 0.12712201348762822, 'subsample': 0.9820523083505434, 'colsample_bytree': 0.917794372996692, 'reg_alpha': 0.6622637858721667, 'reg_la

## Final model

Training XGBoost with Optuna's best parameters (macro F1 0.9915 on
validation, versus 0.9862 with default settings). This is the frozen model
used for the one-time test set evaluation.

In [3]:
best_params = study.best_params
best_params["random_state"] = 42
best_params["eval_metric"] = "mlogloss"

final_model = xgb.XGBClassifier(**best_params)
final_model.fit(X_train, y_train)

print("Final model trained with best parameters")

Final model trained with best parameters


## Test set evaluation

This is the only time the test set is used. It was never seen during
training, validation, or hyperparameter tuning, so this result is the
honest, defensible measure of how the model generalizes.

In [5]:
from sklearn.metrics import classification_report, f1_score

test_df = pd.read_sql("SELECT * FROM sensor_readings WHERE split = 'test'", engine)
X_test = test_df[feature_cols]
y_test = le.transform(test_df["labeling"])

test_preds = final_model.predict(X_test)

print(classification_report(y_test, test_preds, target_names=le.classes_))
print(f"Test macro F1: {f1_score(y_test, test_preds, average='macro'):.4f}")

                              precision    recall  f1-score   support

            Normal condition       1.00      1.00      1.00     23880
Return air temperature fault       0.99      0.99      0.99      1400
Supply air temperature fault       0.99      0.97      0.98       206
            Supply fan fault       1.00      1.00      1.00      9597

                    accuracy                           1.00     35083
                   macro avg       0.99      0.99      0.99     35083
                weighted avg       1.00      1.00      1.00     35083

Test macro F1: 0.9908


In [6]:
train_preds = final_model.predict(X_train)
train_f1 = f1_score(y_train, train_preds, average="macro")
val_f1 = f1_score(y_val, final_model.predict(X_val), average="macro")
test_f1 = f1_score(y_test, test_preds, average="macro")

print(f"Train macro F1: {train_f1:.4f}")
print(f"Val macro F1:   {val_f1:.4f}")
print(f"Test macro F1:  {test_f1:.4f}")

Train macro F1: 1.0000
Val macro F1:   0.9915
Test macro F1:  0.9908


### Checking for overfitting

A perfect training score (1.0000) raised a fair concern about overfitting.
The way to actually test this is comparing train, validation, and test
scores together, not judging train performance in isolation.

| Split | Macro F1 |
|---|---|
| Train | 1.0000 |
| Validation | 0.9915 |
| Test | 0.9908 |

If the model had memorized training data rather than learning a real
pattern, we would expect a large drop from train to validation/test, not
the roughly one-point gap seen here. Validation and test also land within
0.0007 of each other despite being entirely separate, unseen sets. This
is not evidence of overfitting, it reflects that gradient-boosted trees
can fit training data very closely by design, and here that closeness
carried over consistently to genuinely unseen data.

**Final test set result: macro F1 0.9908.** This is the model's honest,
reportable performance, obtained after tuning was already locked in and
without ever adjusting anything based on test set feedback.